# Week 47, Day 2: heapq

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w047d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import heapq

heap = []
for v in [5, 1, 8, 3]:
    heapq.heappush(heap, v)
print(heap, heap[0])
print(heapq.heappop(heap), heapq.heappop(heap), heap)

In [ ]:
import heapq

scores = [72, 95, 64, 88, 91, 70, 85]
heap = scores[:]
heapq.heapify(heap)
print("smallest:", heap[0])
print("pushpop 50:", heapq.heappushpop(heap, 50), "| replace with 50:", heapq.heapreplace(heap, 50), "| top now:", heap[0])
print(heapq.nsmallest(3, scores), heapq.nlargest(2, scores))
names = ["Ifeoma", "Bo", "Kwame", "Ana", "Oluwaseun"]
print(heapq.nlargest(2, names, key=len))

In [ ]:
import heapq

heap = []
for v in [5, 1, 8, 3]:
    heapq.heappush(heap, -v)
print("largest:", -heap[0])
print([-heapq.heappop(heap) for _ in range(len(heap))])

In [ ]:
import heapq

class Reverse:
    def __init__(self, item):
        self.item = item
    def __lt__(self, other):
        return other.item < self.item        # reversed

heap = []
for word in ["pear", "apple", "fig", "kiwi"]:
    heapq.heappush(heap, Reverse(word))
print([heapq.heappop(heap).item for _ in range(4)])

In [ ]:
import heapq, itertools

jobs = []
try:
    heapq.heappush(jobs, (2, {"name": "backup"}))
    heapq.heappush(jobs, (2, {"name": "email"}))
except TypeError as e:
    print("TypeError:", e)

jobs, counter = [], itertools.count()
for priority, name in [(2, "backup"), (1, "deploy"), (2, "email"), (1, "alert")]:
    heapq.heappush(jobs, (priority, next(counter), {"name": name}))
while jobs:
    priority, _, job = heapq.heappop(jobs)
    print(priority, job["name"])

In [ ]:
import heapq, itertools
from dataclasses import dataclass, field

@dataclass(order=True)
class Job:
    priority: int
    seq: int
    name: str = field(compare=False)

counter = itertools.count()
heap = []
for p, name in [(3, "tidy"), (1, "fix"), (3, "plan")]:
    heapq.heappush(heap, Job(p, next(counter), name))
print([heapq.heappop(heap).name for _ in range(3)])

In [ ]:
import heapq, itertools

REMOVED = "<removed>"

class TaskQueue:
    def __init__(self):
        self._heap, self._entries, self._count = [], {}, itertools.count()

    def add(self, task, priority):
        if task in self._entries:
            self.remove(task)                          # changing a priority: retire the old entry
        entry = [priority, next(self._count), task]
        self._entries[task] = entry
        heapq.heappush(self._heap, entry)

    def remove(self, task):
        entry = self._entries.pop(task)
        entry[-1] = REMOVED                            # mark it; it stays in the heap for now

    def pop(self):
        while self._heap:
            priority, _, task = heapq.heappop(self._heap)
            if task is not REMOVED:
                del self._entries[task]
                return task
        raise KeyError("pop from an empty queue")

q = TaskQueue()
for task, p in [("write", 3), ("test", 2), ("ship", 5), ("lunch", 1)]:
    q.add(task, p)
q.add("ship", 0)          # more urgent now
q.remove("lunch")
print([q.pop() for _ in range(3)])

In [ ]:
import heapq

def serve(customers, k):
    tellers = [(0, t) for t in range(k)]           # all free at time 0
    heapq.heapify(tellers)
    for name, arrives, takes in customers:
        free_at, t = heapq.heappop(tellers)
        start = max(arrives, free_at)
        print(f"{name:<6} arrives {arrives:>2}, teller {t} from {start:>2} to {start + takes:>2}, waited {start - arrives}")
        heapq.heappush(tellers, (start + takes, t))

serve([("Ada", 0, 5), ("Bayo", 1, 3), ("Chen", 2, 4), ("Dee", 3, 2), ("Efe", 9, 1)], 2)

> **Key idea.** `heapq` treats a plain list as a min-heap: `heappush`, `heappop`, `heap[0]`, `heapify` (O(n)), `heappushpop` and `heapreplace` (one sift each), `nsmallest` and `nlargest`, and `merge`. For a max-heap, negate numbers, or wrap items in a class with a reversed `__lt__`. Store `(priority, count, item)` so ties never compare items and equal priorities come out first in, first out; a `dataclass(order=True)` with `field(compare=False)` does the same. To remove or change entries, use lazy deletion: mark old entries and skip them when popping. Simulations pop the earliest event from a heap of events, again and again.

### Exercise: The functions

Using `heapq` (no `sort`, `sorted`, `min` or `max`), write `three_smallest(xs)` (a sorted list, by heapifying a copy and popping three times; fewer if `xs` is short), `running_minimums(xs)` (after each push, the smallest so far, using `heap[0]`), and `keep_best(xs, k)`: the k largest values in increasing order, keeping a heap of at most k items and using `heappushpop` once it's full.

In [ ]:
import heapq

def three_smallest(xs):
    ___

def running_minimums(xs):
    ___

def keep_best(xs, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_basics():\n    \"the functions\"\n    assert three_smallest([5, 1, 8, 3, 9]) == [1, 3, 5] and three_smallest([2]) == [2]\n    xs = [5, 1, 8, 3, 9]\n    three_smallest(xs)\n    assert xs == [5, 1, 8, 3, 9], \"work on a copy\"\n    assert running_minimums([5, 7, 2, 4, 1]) == [5, 5, 2, 2, 1]\n    rng = random.Random(5)\n    for _ in range(50):\n        xs = [rng.randint(0, 100) for _ in range(rng.randint(0, 40))]\n        k = rng.randint(1, 10)\n        assert keep_best(xs, k) == sorted(xs)[-k:] if len(xs) >= k else keep_best(xs, k) == sorted(xs)\n    body = source.split(\"def three_smallest\")[1]\n    assert \"sorted(\" not in body and \".sort(\" not in body and \"min(\" not in body and \"max(\" not in body")

### Exercise: Max-heaps two ways

Write `MaxHeap` for numbers (push and pop, storing negated values in a `heapq` list; `peek` and `len` too), and `largest_words(words, k)`: the k alphabetically **last** words, largest first, using a `heapq` heap of `Reverse` wrappers (write the class) and k pops. No sorting.

In [ ]:
import heapq

class MaxHeap:
    ___

class Reverse:
    ___

def largest_words(words, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_max():\n    \"max-heaps\"\n    h = MaxHeap()\n    for v in [5, 1, 8, 3, -2]:\n        h.push(v)\n    assert h.peek() == 8 and len(h) == 5 and [h.pop() for _ in range(5)] == [8, 5, 3, 1, -2]\n    words = [\"pear\", \"apple\", \"fig\", \"kiwi\", \"plum\", \"date\"]\n    assert largest_words(words, 3) == [\"plum\", \"pear\", \"kiwi\"] and largest_words(words, 0) == []\n    assert Reverse(\"b\") < Reverse(\"a\") and not (Reverse(\"a\") < Reverse(\"b\"))\n    assert \"sort\" not in source")

### Exercise: A fair priority queue

Write `PriorityQueue` around `heapq`: `push(item, priority)` for **any** items (dicts included), `pop()` returning the item with the smallest priority, first in first out among equal priorities (`IndexError` when empty), `peek()`, `len`, and `drain()`: pop everything, returning a list.

In [ ]:
import heapq, itertools

class PriorityQueue:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_pq():\n    \"fair and safe\"\n    q = PriorityQueue()\n    for p, name in [(2, \"backup\"), (1, \"deploy\"), (2, \"email\"), (1, \"alert\"), (2, \"audit\")]:\n        q.push({\"name\": name}, p)\n    assert len(q) == 5 and q.peek() == {\"name\": \"deploy\"}\n    assert [j[\"name\"] for j in q.drain()] == [\"deploy\", \"alert\", \"backup\", \"email\", \"audit\"] and len(q) == 0\n    try:\n        q.pop()\n        assert False\n    except IndexError:\n        pass\n    rng = random.Random(6)\n    q, ref = PriorityQueue(), []\n    for i in range(500):\n        p = rng.randint(0, 5)\n        q.push([i], p)\n        ref.append((p, i))\n    assert [x[0] for x in q.drain()] == [i for p, i in sorted(ref)]")

### Exercise: Lazy deletion

Write `TaskQueue` with `add(task, priority)` (adding or changing a priority), `remove(task)` (`KeyError` if absent), `pop()` (the most urgent live task; `KeyError` when none), `len` (live tasks only) and `task in q`, using lazy deletion. Also `compact()`, which rebuilds the heap from live entries only, and make `add` and `remove` call it automatically whenever the heap holds more than twice as many entries as there are live tasks, plus 10, so it can't grow without limit.

In [ ]:
import heapq, itertools

REMOVED = "<removed>"

class TaskQueue:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_tasks():\n    \"tasks\"\n    q = TaskQueue()\n    for task, p in [(\"write\", 3), (\"test\", 2), (\"ship\", 5), (\"lunch\", 1)]:\n        q.add(task, p)\n    q.add(\"ship\", 0)\n    q.remove(\"lunch\")\n    assert len(q) == 3 and \"lunch\" not in q and \"ship\" in q\n    assert [q.pop() for _ in range(3)] == [\"ship\", \"test\", \"write\"] and len(q) == 0\n    for f in (q.pop, lambda: q.remove(\"nothing\")):\n        try:\n            f()\n            assert False\n        except KeyError:\n            pass\ndef test_random():\n    \"against a dict, with bounded growth\"\n    rng = random.Random(7)\n    q, ref = TaskQueue(), {}\n    order = itertools.count()\n    for _ in range(6000):\n        t = rng.randrange(40)\n        r = rng.random()\n        if r < 0.6:\n            p = rng.randint(0, 9)\n            q.add(t, p)\n            ref[t] = (p, next(order))\n        elif r < 0.8 and t in ref:\n            q.remove(t)\n            del ref[t]\n        elif ref:\n            best = min(ref, key=lambda k: ref[k])\n            assert q.pop() == best\n            del ref[best]\n        assert len(q) == len(ref)\n        heap = [v for v in vars(q).values() if isinstance(v, list)]\n        assert heap and len(heap[0]) <= 2 * len(ref) + 11, \"compact the heap\"")

### Exercise: The bank

Write `serve(customers, k)`: customers are `(name, arrives, takes)` tuples in order of arrival; there are k tellers numbered 0 to k − 1, all free at time 0. Each customer goes to the teller who has been free the longest (smallest free time, ties to the lower number), starting at `max(arrives, free_at)`. Return a list of `(name, teller, start, finish)`, then write `average_wait(customers, k)` (0.0 for no customers) and `fewest_tellers(customers, max_wait)`: the smallest k for which nobody waits longer than `max_wait`.

In [ ]:
import heapq

def serve(customers, k):
    ___

def average_wait(customers, k):
    ___

def fewest_tellers(customers, max_wait):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bank():\n    \"tellers\"\n    cs = [(\"Ada\", 0, 5), (\"Bayo\", 1, 3), (\"Chen\", 2, 4), (\"Dee\", 3, 2), (\"Efe\", 9, 1)]\n    assert serve(cs, 2) == [(\"Ada\", 0, 0, 5), (\"Bayo\", 1, 1, 4), (\"Chen\", 1, 4, 8), (\"Dee\", 0, 5, 7), (\"Efe\", 0, 9, 10)]\n    assert average_wait(cs, 2) == (0 + 0 + 2 + 2 + 0) / 5 and average_wait([], 3) == 0.0\n    assert average_wait(cs, 1) == (0 + 4 + 6 + 9 + 5) / 5\n    assert fewest_tellers(cs, 0) == 4 and fewest_tellers(cs, 2) == 2 and fewest_tellers(cs, 100) == 1\n    assert serve([(\"a\", 0, 1), (\"b\", 0, 1), (\"c\", 5, 1)], 3) == [(\"a\", 0, 0, 1), (\"b\", 1, 0, 1), (\"c\", 2, 5, 6)]")

### Bug hunt: A heap that wasn't

This pops from a list that was never made into a heap, so `heappop` trusts `heap[0]` to be the smallest and it isn't: this prints `5`. Make the list a heap first, so it prints `1`.

In [ ]:
import heapq

heap = [5, 3, 1, 4]
print(heapq.heappop(heap))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"heapify first\"\n    expect_output(\"1\")")

### Bug hunt: Comparing what can't be compared

Two jobs have the same priority, so the tuples' second items, dicts, get compared, and this crashes with a `TypeError`. Add a counter between priority and job so it prints `backup email`.

In [ ]:
import heapq

heap = []
heapq.heappush(heap, (2, {"name": "backup"}))
heapq.heappush(heap, (2, {"name": "email"}))
print(heapq.heappop(heap)[-1]["name"], heapq.heappop(heap)[-1]["name"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a tie-breaker\"\n    expect_output(\"backup email\")")

### Bug hunt: Still negative

This max-heap negates values going in but forgets to negate them coming out: this prints `[-8, -5, -1]`. Negate on the way out so it prints `[8, 5, 1]`.

In [ ]:
import heapq

heap = []
for v in [5, 1, 8]:
    heapq.heappush(heap, -v)
print([heapq.heappop(heap) for _ in range(3)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"negate back\"\n    expect_output(\"[8, 5, 1]\")")

### Puzzle: Popping past removed entries

Arrange the lines into the lazy-deletion `pop`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import heapq

h = [4, 7, 9]          # already a heap

p = ___   # heapq.heappushpop(h, 2)  (h is unchanged afterwards)
q = ___   # heapq.heapreplace(h, 2)  (on the original h)
r = ___   # heapq.nlargest(2, [3, 9, 1, 7])
s = ___   # heapq.nsmallest(2, ["bb", "a", "ccc"], key=len)
t = ___   # -heapq.heappop(x) where x = [-9, -4, -7]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 4 and r == [9, 7], \"p, q and r\"\n    assert s == [\"a\", \"bb\"] and t == 9, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a job scheduler

A single machine runs jobs one at a time and never interrupts one. Each job is `(name, arrives, takes, priority)`; a smaller priority number is more urgent. Whenever the machine is free, it starts the most urgent job that has **already arrived** (ties: earlier arrival, then name); if none has arrived, it sits idle until the next arrival. Write `schedule(jobs)` returning `[(name, start, finish), ...]` in the order the jobs run.

Sort the jobs by arrival once, then keep a `heapq` heap of the jobs that have arrived and are waiting: before each choice, push every job whose arrival time has come. Also write `waits(jobs)`: a dict from name to how long the job waited before starting. Jobs can arrive in any order in the input; the checks run 20,000 jobs, so each choice must be O(log n).

In [ ]:
import heapq

def schedule(jobs):
    ___

def waits(jobs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow(jobs):\n    left, t, out = list(jobs), 0, []\n    while left:\n        ready = [j for j in left if j[1] <= t]\n        if not ready:\n            t = min(j[1] for j in left)\n            continue\n        j = min(ready, key=lambda j: (j[3], j[1], j[0]))\n        left.remove(j)\n        out.append((j[0], t, t + j[2]))\n        t += j[2]\n    return out\ndef test_small():\n    \"examples\"\n    jobs = [(\"report\", 0, 4, 2), (\"alert\", 1, 1, 0), (\"backup\", 2, 3, 2), (\"email\", 2, 1, 1), (\"late\", 20, 2, 5)]\n    assert schedule(jobs) == [(\"report\", 0, 4), (\"alert\", 4, 5), (\"email\", 5, 6), (\"backup\", 6, 9), (\"late\", 20, 22)]\n    assert waits(jobs) == {\"report\": 0, \"alert\": 3, \"email\": 3, \"backup\": 4, \"late\": 0}\n    assert schedule([]) == [] and schedule(list(reversed(jobs))) == schedule(jobs)\ndef test_random():\n    \"against a slow version\"\n    rng = random.Random(8)\n    for _ in range(100):\n        jobs = [(f\"j{i}\", rng.randint(0, 30), rng.randint(1, 6), rng.randint(0, 3)) for i in range(rng.randint(0, 15))]\n        assert schedule(jobs) == slow(jobs)\ndef test_big():\n    \"fast enough\"\n    rng = random.Random(9)\n    jobs = [(f\"j{i:05}\", rng.randint(0, 50_000), rng.randint(1, 5), rng.randint(0, 9)) for i in range(20_000)]\n    t0 = time.perf_counter()\n    out = schedule(jobs)\n    assert len(out) == 20_000 and time.perf_counter() - t0 < 3")

## Warm-up

`heapq` has no heap class: the heap **is** the list, and the functions keep it in heap order (the same order you'd have got yesterday). `heap[0]` is the smallest item, and you mustn't change the list by any other means, or the order breaks.

## The heapq functions

| function | does | cost |
|---|---|---|
| `heappush(h, x)` | push | O(log n) |
| `heappop(h)` | pop the smallest (`IndexError` if empty) | O(log n) |
| `h[0]` | look at the smallest | O(1) |
| `heapify(xs)` | turn a list into a heap, in place | O(n) |
| `heappushpop(h, x)` | push x, then pop the smallest (faster than both) | O(log n) |
| `heapreplace(h, x)` | pop the smallest, then push x | O(log n) |
| `nsmallest(k, xs, key=)`, `nlargest(k, xs, key=)` | the k smallest or largest, sorted | about O(n log k) |
| `merge(*sorted_iterables, key=)` | merge sorted inputs lazily | O(n log k) |

`heappushpop` and `heapreplace` differ in order: `heappushpop` may hand back the item you just gave it (if it's smaller than everything), while `heapreplace` always pops the old top first, so it can return something larger than what you pushed. Both are a single sift, which is why they're faster than a push and a pop.

For `k == 1`, `min` and `max` are simpler; when k is close to n, `sorted(xs)[:k]` is quicker. `nsmallest` and `nlargest` win in between, especially when n is large and k small (Day 4 shows how they work).

## Max-heaps

`heapq` only makes **min**-heaps. For a max-heap of numbers, push `-x` and negate what you pop: the largest x has the smallest −x.

Negation only works for numbers. For anything else (strings, tuples, dates), wrap each item in a small class whose `<` is reversed; `heapq` uses only `<`:

## Ties and things that can't be compared

Priority queues usually hold `(priority, item)` tuples. Tuples compare element by element, so when two priorities are **equal**, Python compares the items, and that goes wrong in two ways:

- If the items can't be compared (dicts, most objects), you get a `TypeError` in the middle of a push or pop.
- Even when they can, equal-priority items come out in the items' own order, not the order they arrived in. A to-do list usually wants first-come, first-served among equals.

The fix: put a **counter** between them, `(priority, count, item)`. Counts are all different, so the comparison never reaches the item, and among equal priorities the earlier count wins: first in, first out. `itertools.count()` hands out 0, 1, 2, ... each time you call `next` on it.

A **dataclass** (Week 32) can do the same job more readably: `@dataclass(order=True)` compares fields in order, and `field(compare=False)` leaves the item out of comparisons.

## Changing your mind: lazy deletion

A heap can't find an entry quickly, so it can't easily remove one or change its priority (yesterday's challenge kept an index map to do it). The simpler trick, from the "priority queue implementation notes" in Python's `heapq` documentation, is **lazy deletion**: keep a dict from each item to its current entry; to remove an item, **mark** its entry as removed and leave it in the heap; to change a priority, mark the old entry and push a new one. When popping, skip marked entries.

Entries are lists, not tuples, so their last slot can be changed. Marked entries take space until they reach the top; if many pile up, rebuild the heap from the live entries now and then.

## Simulations

In a **discrete-event simulation**, the program keeps a heap of future **events** ordered by time, and repeatedly pops the earliest, handles it (which may schedule more events), and moves the clock to its time. Nothing happens between events, so the simulation jumps straight from one to the next, however far apart they are.

A simple case doesn't even need event objects. Customers arrive at a bank with k tellers: each takes the teller who's been free the longest. Keep a heap of `(free_at, teller)`:

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a queue you've stood in where some people were served first (a hospital's emergency department, boarding a plane). What's the priority, and how are ties broken? Tomorrow: heap sort, which sorts a list in place with the heap you built yesterday.